In [1]:
import chess.svg
import einops
import numpy as np
from pathlib import Path

from chess_gnn.inference import ChessBoardPredictor
from chess_gnn.models import *
from chess_gnn.utils import PGNBoardHelper, ChessPoint

import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objects as go
import chess
import chess.svg
import base64

In [2]:
class AttentionMapGetter:
    def __init__(self, attn_matrix: np.ndarray):
        self.attn_matrix = attn_matrix
    
    def get_attention_map(self, layer_idx: int, head_idx: int, query: str):
        if query.lower() == 'cls':
            point_idx = 0
        else:
            point = ChessPoint.from_square(query)
            point_idx = point.to_str_position() + 1
    
        attention = np.flipud(einops.rearrange(self.attn_matrix[layer_idx][head_idx][point_idx][1:], '(h w) -> h w', h=8))
        return attention
        

In [3]:
def create_game_attention_app(boards: list[chess.Board], attn_maps: list[AttentionMapGetter]):
    assert len(boards) == len(attn_maps), "Boards and attention maps must be same length"

    app = dash.Dash(__name__)
    num_moves = len(boards)
    default_square = "a1"
    default_idx = 0
    default_layer = 0
    default_head = 0

    example_map = attn_maps[0].get_attention_map(default_layer, default_head, default_square)
    num_layers = attn_maps[0].attn_matrix.shape[0]
    num_heads = attn_maps[0].attn_matrix.shape[1]

    def get_svg_images(board: chess.Board):
        images = []
        for idx in range(64):
            piece = board.piece_at(idx)
            point = ChessPoint.from_1d(idx)
            if piece:
                svg = chess.svg.piece(piece)
                svg_bytes = svg.encode('utf-8')
                uri = f"data:image/svg+xml;base64,{base64.b64encode(svg_bytes).decode('utf-8')}"
                images.append(dict(
                    source=uri,
                    xref="x", yref="y",
                    x=point.x,
                    y=point.y,
                    sizex=0.9, sizey=0.9,
                    xanchor="center", yanchor="middle",
                    layer="above"
                ))
        return images

    def create_figure(board: chess.Board, attention: np.ndarray, highlight_square: str | None = None):
        fig = go.Figure(data=go.Heatmap(
            z=attention,
            x=list(range(8)),
            y=list(range(8)),
            colorscale='Viridis',
            hoverongaps=False,
            opacity=0.5
        ))
        fig.update_layout(
            yaxis=dict(scaleanchor="x", scaleratio=1),
            xaxis=dict(constrain='domain'),
            images=get_svg_images(board),
            shapes=[]
        )

        if highlight_square and highlight_square != 'cls':
            col, row = ChessPoint.from_square(highlight_square)
            fig.update_layout(shapes=[dict(
                type="rect",
                x0=col - 0.5, y0=row - 0.5,
                x1=col + 0.5, y1=row + 0.5,
                line=dict(color="red", width=1),
                fillcolor="rgba(255,0,0,0.2)",
                layer="above"
            )])
        elif highlight_square == 'cls':
            fig.update_layout(shapes=[dict(
                type="rect",
                x0=-0.5, y0=-1,
                x1=7.5, y1=-0.6,
                line=dict(color="red", width=2),
                fillcolor="rgba(255,0,0,0.2)",
                layer="above"
            )])

        return fig

    app.layout = html.Div([
        html.Div([
            html.Button("Prev", id="prev-btn", n_clicks=0),
            html.Button("Next", id="next-btn", n_clicks=0),
            html.Span(id="move-label", style={"marginLeft": "1rem"}),
        ], style={"marginBottom": "1rem"}),

        dcc.Store(id="current-index", data=0),
        dcc.Store(id="current-square", data=default_square),
        dcc.Store(id="current-layer", data=default_layer),
        dcc.Store(id="current-head", data=default_head),
        
        html.Div(id='click-output', style={
            "textAlign": "center",
            "fontWeight": "bold",
            "marginBottom": "0.5rem"
        }),

        dcc.Graph(
            id='heatmap',
            figure=create_figure(boards[default_idx], example_map),
            config={"displayModeBar": False},
            style={"width": "100%", "height": "600px"}
        ),

        html.Div([
            html.Label("Layer"),
            dcc.Slider(
                id='layer-slider',
                min=0,
                max=num_layers - 1,
                step=1,
                value=default_layer,
                marks={i: str(i) for i in range(num_layers)},
                tooltip={"placement": "bottom", "always_visible": True},
            ),
        ], style={"marginTop": "1rem"}),

        html.Div([
            html.Label("Head"),
            dcc.Slider(
                id='head-slider',
                min=0,
                max=num_heads - 1,
                step=1,
                value=default_head,
                marks={i: str(i) for i in range(num_heads)},
                tooltip={"placement": "bottom", "always_visible": True},
            ),
        ], style={"marginBottom": "1rem"}),

        html.Div([
            html.Button("CLS", id="cls-btn", n_clicks=0, style={"marginRight": "10px"})
        ], style={
            "position": "absolute",
            "top": "10px",
            "right": "10px",
            "zIndex": 10,
            "display": "flex",
            "alignItems": "center"
        }),
    ], style={"position": "relative", "paddingTop": "2rem"})

    @app.callback(
        Output('current-index', 'data'),
        Output('move-label', 'children'),
        Input('prev-btn', 'n_clicks'),
        Input('next-btn', 'n_clicks'),
        State('current-index', 'data')
    )
    def update_index(prev, nxt, current):
        ctx = dash.callback_context.triggered_id
        if ctx == 'prev-btn':
            current = max(0, current - 1)
        elif ctx == 'next-btn':
            current = min(num_moves - 1, current + 1)

        move_dict = {0: 'White', 1: 'Black'}
        to_move = current % 2
        return current, f"Move: {current}/{num_moves - 1}     {move_dict[to_move]} to move"

    @app.callback(
        Output('current-square', 'data'),
        Output('click-output', 'children'),
        Input('heatmap', 'clickData'),
        Input('cls-btn', 'n_clicks'),
        State('cls-btn', 'n_clicks_timestamp'),
        State('heatmap', 'clickData'),
        State('current-square', 'data'),
        prevent_initial_call=True
    )
    def update_square(clickData, cls_clicks, cls_ts, heatmap_click, current_square):
        ctx = dash.callback_context
        if ctx.triggered_id == 'cls-btn':
            return 'cls', "Viewing attention from: cls"
        elif ctx.triggered_id == 'heatmap' and clickData and 'points' in clickData:
            point = clickData['points'][0]
            col = int(point['x'])
            row = int(point['y'])
            file = chr(ord('a') + col)
            rank = row + 1
            square = f"{file}{rank}"
            return square, f"Viewing attention from: {square}"
        return current_square, f"Viewing attention from: {current_square}"

    @app.callback(
        Output('current-layer', 'data'),
        Input('layer-slider', 'value')
    )
    def update_layer_index(val):
        return val

    @app.callback(
        Output('current-head', 'data'),
        Input('head-slider', 'value')
    )
    def update_head_index(val):
        return val

    @app.callback(
        Output('heatmap', 'figure'),
        Input('current-index', 'data'),
        Input('current-square', 'data'),
        Input('current-layer', 'data'),
        Input('current-head', 'data')
    )
    def update_figure(move_idx, square, layer_idx, head_idx):
        board = boards[move_idx]
        attention = attn_maps[move_idx].get_attention_map(layer_idx, head_idx, square)
        return create_figure(board, attention, highlight_square=square)

    return app


In [7]:
pgn = PGNBoardHelper(Path('/Users/ray/Datasets/chess/Carlsen.pgn'))

ckpt_file = '/Users/ray/models/chess/transformer/f7cd6575-69f5-4f75-aae0-7488b17492ce/epoch=0-step=20000.ckpt'
model = ChessTransformer.load_from_checkpoint(ckpt_file)

encoder = model.get_encoder()
predictor = ChessBoardPredictor(encoder=encoder)

for i in range(1729):
    pgn.get_game()
    
board_fens = pgn.get_board_fens()

boards_in = []
attns_in = []
for i, board_fen in enumerate(board_fens):
    board = chess.Board(board_fen)
    attn_matrix = predictor.get_attn(chess_board=board)
    attn = AttentionMapGetter(attn_matrix)
    boards_in.append(board)
    attns_in.append(attn)

/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'encoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder'])`.
/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'decoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['decoder'])`.
/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'mask_handler' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['mask_handler'])`.


KeyboardInterrupt: 

In [ ]:
app = create_game_attention_app(boards_in, attns_in)
app.run(mode="jupyter-inline", port=8052)